<a href="https://colab.research.google.com/github/adishup/gen-ai-lab/blob/main/experiment%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers sentencepiece accelerate torch

In [ ]:
# ============================================================
# EXPERIMENT 3
# TEXT SUMMARIZATION AND QUESTION ANSWERING
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering
)


# ============================================================
# 1. SELECT DEVICE
# ============================================================

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("GPU Available")
    print("GPU:", torch.cuda.get_device_name(0))

else:

    device = torch.device("cpu")

    print("GPU not available. Using CPU.")


# ============================================================
# PART A — TEXT SUMMARIZATION
# ============================================================

print("\n" + "=" * 60)
print("PART A - TEXT SUMMARIZATION")
print("=" * 60)


# ------------------------------------------------------------
# 2. Load BART Tokenizer
# ------------------------------------------------------------

summarization_model_name = "facebook/bart-large-cnn"

print("\nLoading BART summarization model...")

summarization_tokenizer = AutoTokenizer.from_pretrained(
    summarization_model_name
)


# ------------------------------------------------------------
# 3. Load BART Model
# ------------------------------------------------------------

summarization_model = AutoModelForSeq2SeqLM.from_pretrained(
    summarization_model_name
)

summarization_model = summarization_model.to(device)

summarization_model.eval()

print("BART model loaded successfully.")


# ------------------------------------------------------------
# 4. Input Text
# ------------------------------------------------------------

text = """
Artificial Intelligence is transforming many industries by
enabling machines to perform tasks that normally require human
intelligence. It is widely used in healthcare, education,
manufacturing, finance, transportation, and cybersecurity.
AI systems can analyze large amounts of data, identify patterns,
make predictions, and support intelligent decision-making.
Generative AI is a branch of Artificial Intelligence that can
create new content such as text, images, audio, video, and
computer programs.
"""


# ------------------------------------------------------------
# 5. Tokenize Text
# ------------------------------------------------------------

inputs = summarization_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)


# Move inputs to GPU/CPU
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)


# ------------------------------------------------------------
# 6. Generate Summary
# ------------------------------------------------------------

print("\nGenerating summary...")

with torch.no_grad():

    summary_ids = summarization_model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=60,
        min_length=20,
        num_beams=4,
        do_sample=False
    )


# ------------------------------------------------------------
# 7. Decode Summary
# ------------------------------------------------------------

summary = summarization_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)


# ------------------------------------------------------------
# 8. Display Summary
# ------------------------------------------------------------

print("\nOriginal Text:")
print(text)

print("\nSummary:")
print(summary)


# ============================================================
# PART B — QUESTION ANSWERING
# ============================================================

print("\n" + "=" * 60)
print("PART B - QUESTION ANSWERING")
print("=" * 60)


# ------------------------------------------------------------
# 9. Load Question Answering Model
# ------------------------------------------------------------

qa_model_name = "distilbert-base-cased-distilled-squad"

print("\nLoading Question Answering model...")

qa_tokenizer = AutoTokenizer.from_pretrained(
    qa_model_name
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
)

qa_model = qa_model.to(device)

qa_model.eval()

print("Question Answering model loaded successfully.")


# ------------------------------------------------------------
# 10. Context
# ------------------------------------------------------------

context = """
Generative Artificial Intelligence is a type of Artificial
Intelligence that can create new content such as text, images,
audio, video, and computer programs. Large Language Models are
commonly used for text generation, summarization, translation,
and question answering.
"""


# ------------------------------------------------------------
# 11. Question
# ------------------------------------------------------------

question = "What type of content can Generative AI create?"


# ------------------------------------------------------------
# 12. Tokenize Question and Context
# ------------------------------------------------------------

qa_inputs = qa_tokenizer(
    question,
    context,
    return_tensors="pt",
    truncation=True
)


# Move inputs to GPU/CPU
qa_inputs = {
    key: value.to(device)
    for key, value in qa_inputs.items()
}


# ------------------------------------------------------------
# 13. Predict Answer
# ------------------------------------------------------------

print("\nFinding answer...")

with torch.no_grad():

    qa_outputs = qa_model(
        **qa_inputs
    )


# ------------------------------------------------------------
# 14. Find Start and End Positions
# ------------------------------------------------------------

start_position = torch.argmax(
    qa_outputs.start_logits
)

end_position = torch.argmax(
    qa_outputs.end_logits
)


# Make sure end is not before start
if end_position < start_position:
    end_position = start_position


# ------------------------------------------------------------
# 15. Extract Answer Tokens
# ------------------------------------------------------------

answer_tokens = qa_inputs["input_ids"][
    0,
    start_position:end_position + 1
]


# ------------------------------------------------------------
# 16. Convert Tokens to Text
# ------------------------------------------------------------

answer = qa_tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)


# ------------------------------------------------------------
# 17. Calculate Confidence Score
# ------------------------------------------------------------

start_score = torch.softmax(
    qa_outputs.start_logits,
    dim=-1
)[0, start_position]

end_score = torch.softmax(
    qa_outputs.end_logits,
    dim=-1
)[0, end_position]

confidence = (
    start_score * end_score
).item()


# ------------------------------------------------------------
# 18. Display Answer
# ------------------------------------------------------------

print("\nContext:")
print(context)

print("\nQuestion:")
print(question)

print("\nAnswer:")
print(answer)

print("\nConfidence Score:")
print(round(confidence, 3))


# ============================================================
# 19. COMPLETION
# ============================================================

print("\n" + "=" * 60)
print("EXPERIMENT 3 COMPLETED SUCCESSFULLY")
print("=" * 60)

GPU not available. Using CPU.

PART A - TEXT SUMMARIZATION

Loading BART summarization model...


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Question Answering model loaded successfully.

Finding answer...

Context:

Generative Artificial Intelligence is a type of Artificial
Intelligence that can create new content such as text, images,
audio, video, and computer programs. Large Language Models are
commonly used for text generation, summarization, translation,
and question answering.


Question:
What type of content can Generative AI create?

Answer:
new content such as text, images, audio, video, and computer programs

Confidence Score:
0.267

EXPERIMENT 3 COMPLETED SUCCESSFULLY
